# RAG and Agent Evaluation Introduction

Sources:
1. Introduction: 
https://www.youtube.com/watch?v=VKHBP0QSCFo&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv 1.

2. Prepare Data
https://www.youtube.com/watch?v=utkcclfpj0g&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv

3. 
1. We have ground_truth where we have the question, and document_id,we can get the correct answer from the documents A,
2. we give the question from the ground truth to RAG, then it generates for us A' 
3. answer A and A' maybe have the same meaning but they are different in words, so we will use another RAG to judge. The judge RAG classifies each answer as good or bad and explains its reasoning. We can ask the LLM judge why it made a decision generally produces better classifications than asking for just a verdict (judge).

# Prepare Data for LLM Judge

In [2]:
import sys
import os
import json
import pandas as pd
from rich import print
from minsearch import Index

parent_directory = os.path.dirname(os.getcwd())
sys.path.append(parent_directory)
from evaluation_utils import compute_relevance_total, hit_rate, mean_reciprocal_rank
from ingest import (load_faq_data,
                    build_index,
                    text_search
)
# load the documents, fillter llm_zoocamp ONLY since the ground_truth data is generated from the llm_zoomcamp ONLY for simplicity
documents = load_faq_data()
documents_llm = [doc for doc in documents if doc['course'] == "llm-zoomcamp"]
print(f'The number of documents = {len(documents_llm)}')

# get the created ground-truth data
df_ground_truth = pd.read_csv('./data/ground_truth_FAQ_data.csv')
ground_truth = df_ground_truth.to_dict(orient="records")
# convert the df to dictionary

# note some answers do not have 5 questions
print(f'The number of ground_truth = {len(ground_truth)}')

# create search index and fit with document
search_index = build_index(documents_llm)

# Bind your fitted index to the search function using a lambda
bound_text_search = lambda query: text_search(query=query, search_index=search_index)

The number of documents = 113

The number of ground_truth = 510

In [3]:
documents_llm[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [4]:
# create a dictionary, each key is the document_id that has value document
documents_llm_idx = dict()

for doc in documents_llm:
    if doc['id'] not in documents_llm_idx.keys():
        documents_llm_idx[doc['id']]= doc
    else:
        print(f'The document id: {documents_llm['id']} is available')
        continue

In [5]:
from evaluation_utils import RAGWithUsage, run_in_parallel
asssistant_with_usage = RAGWithUsage(index=search_index) #  this rag is using a search index with the optimal parameters
asssistant_with_usage.rag(query="if I go for a holiday and I can not submit the final module, can I get a certificate")

'No, you can only get a certificate if you finish the course with a "live" cohort. You need to peer-review 3 capstone(s) after submitting your project.'

In [5]:
asssistant_with_usage.get_last_tokens()

{'input_tokens': 1390, 'output_tokens': 39}

In [6]:
asssistant_with_usage.get_total_tokens()

{'total_input_tokens': 1390, 'total_output_tokens': 39}

## get answer from rag for a question from ground truth

In [18]:
rec = ground_truth[0]
rec

{'question': "Okay, so I’m really confused about Office Hours. Where do I actually find the Zoom link? It's not publicly available, right?",
 'document': '489dd1c9d9'}

In [7]:
record_in_ground_truth = ground_truth[0]
def generate_rag_answer(record_in_ground_truth):

    question =  record_in_ground_truth['question']
    document = record_in_ground_truth['document']
    answer_orig = documents_llm_idx[document]['answer']
    answer_llm = asssistant_with_usage.rag(question)

    result = {
            "question": question,
            "answer_llm": answer_llm,
            "answer_orig": answer_orig,
            "document": document,
        }
    return result

generate_rag_answer(record_in_ground_truth)

{'question': "Okay, so I’m really confused about Office Hours. Where do I actually find the Zoom link? It's not publicly available, right?",
 'answer_llm': 'The zoom link is only published to instructors/presenters/TAs.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.',
 'document': '489dd1c9d9'}

In [8]:
# create a function
from evaluation_utils import run_in_parallel, generate_ground_truth


# you need to reset the token before run in parallel

asssistant_with_usage = RAGWithUsage(index=search_index)
results = run_in_parallel(
    func=generate_rag_answer,
    items=ground_truth,
    max_workers=6
)

Processing:   0%|          | 0/510 [00:00<?, ?it/s]

Processing: 100%|██████████| 510/510 [21:12<00:00,  2.49s/it]


In [9]:
# Calculate the total input and output token since I use ollama only
asssistant_with_usage.get_total_tokens()

{'total_input_tokens': 1198052, 'total_output_tokens': 32349}

In [10]:
results[0]

{'question': "I noticed you mentioned watching live on DataTalksClub's YouTube Channel. Is that the *only* place to see the recordings after they happen, or will there be other options?",
 'answer_llm': "The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub). Don’t post questions in chat as they may be missed if the room is very active. I don't know.",
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be

In [11]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [ ]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("./data/question_with_rag_and_human_answers_data.csv", index=False)

In [13]:
df_answers.shape

(510, 4)

In [14]:
df_answers.iloc[9]

question       The homework is similar to the lessons but wit...
answer_llm     Working on the material or homework in advance...
answer_orig    Start with the [LLM Zoomcamp docs](https://dat...
document                                              04919992b3
Name: 9, dtype: str